# Transfer Learning and Distillation Experiments

This notebook implements VGG16 feature extraction and knowledge distillation using the tuned baseline architecture as the student.


In [ ]:
import os
import sys
import json
import random
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers

from helpers import data_utils, model_utils, training_utils, visualization

keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
USE_COLAB = True
USE_KAGGLE = False  # Set True when running on Kaggle

In [ ]:
# Configuration
if USE_KAGGLE:
    DATASET_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/"
    SAVE_DIR = pathlib.Path("/kaggle/working/saved_models")
elif USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = "/content/drive/MyDrive/x-ray-dataset/"
    SAVE_DIR = pathlib.Path("/content/drive/MyDrive/saved_models")
else:
    DATASET_ROOT = "dataset"
    SAVE_DIR = pathlib.Path("saved_models")

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.1
AUTOTUNE   = tf.data.AUTOTUNE

for split_path in [TRAIN_DIR, TEST_DIR]:
    if not os.path.isdir(split_path):
        raise FileNotFoundError(f"Missing directory: {split_path}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model checkpoint paths
TEACHER_CHECKPOINT_PATH  = SAVE_DIR / "vgg16_frozen_best_checkpoint.keras"
FINETUNE_CHECKPOINT_PATH = SAVE_DIR / "vgg16_finetuned_best_checkpoint.keras"

runtime_name = "Kaggle" if USE_KAGGLE else ("Google Colab" if USE_COLAB else "Local")
print(f"Environment:  {runtime_name}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Save dir:     {SAVE_DIR}")


In [ ]:
# Bind a notebook-local saver without redefining utility functions here.
save_model_with_meta = training_utils.make_model_saver(SAVE_DIR)


In [ ]:
# Gather train(+optional original val) and test paths/labels
all_train_paths, all_train_labels, test_paths, test_labels = data_utils.gather_split_paths_labels(
    TRAIN_DIR,
    TEST_DIR,
    val_dir=VAL_DIR,
)

if len(np.unique(all_train_labels)) < 2:
    raise ValueError("Both NORMAL and PNEUMONIA classes must exist in the training pool.")

print(f"Training pool size (train + optional original val): {len(all_train_paths)} images")
print(f"  NORMAL={(all_train_labels==0).sum()}, PNEUMONIA={(all_train_labels==1).sum()}")
print(f"Test set size: {len(test_paths)} images")


Training pool size (train + optional original val): 5232 images
  NORMAL=1349, PNEUMONIA=3883
Test set size: 624 images


In [ ]:
data_augmentation = data_utils.build_data_augmentation(
    rotation=30,
    width_shift=0.1,
    height_shift=0.1,
    shear=0.2,
    zoom=0.2,
)

# Build train/val/test datasets with raw [0, 255] images.
# VGG16 preprocessing is applied internally by the model after augmentation —
# do NOT pass a preprocess_fn here to avoid double-preprocessing.
train_ds, val_ds, test_ds, ds_meta = data_utils.build_train_val_test_datasets(
    all_train_paths,
    all_train_labels,
    test_paths,
    test_labels,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    seed=42,
    autotune=AUTOTUNE,
)

# Paper-aligned class weights: penalise missed pneumonia cases 4x
# (clinical priority to minimise false negatives), matching the paper's
# explicit value of weight=4 for the PNEUMONIA class.
class_weights = {0: 1.0, 1: 4.0}

print(f"Train split -> NORMAL={ds_meta['normal_count']}, PNEUMONIA={ds_meta['pneumonia_count']}")
print(f"Validation size: {len(ds_meta['val_labels'])}")
print("Class weights:", class_weights)
print("Datasets are ready.")


## VGG16 Training

In [ ]:
# ── Phase 1: Feature extraction with frozen VGG16 base ──────────────────────
# Raw [0,255] datasets are used; the model applies VGG16 preprocessing
# internally after augmentation (no double-preprocessing).

teacher_model = model_utils.build_vgg16_model(
    img_size=IMG_SIZE,
    augmentation_layer=data_augmentation,
    dense_units=256,
    dense_units_2=128,
    dropout=0.5,
    dropout_2=0.3,
    learning_rate=1e-4,
    freeze_base=True,
    name="vgg16_frozen",
)
teacher_model.summary(show_trainable=True)

teacher_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=TEACHER_CHECKPOINT_PATH,
    patience=7,
    monitor="val_auc",
)

teacher_history = teacher_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=teacher_callbacks,
    class_weight=class_weights,
    verbose=1,
)
print(f"Phase 1 complete. Best checkpoint saved to: {TEACHER_CHECKPOINT_PATH}")


In [ ]:
# ── Phase 2: Fine-tuning — unfreeze VGG16 block5 ────────────────────────────
# Load the best Phase-1 checkpoint and unfreeze the last conv block
# (block5_conv1/2/3 + block5_pool) with a 10x smaller learning rate.

# Plot Phase 1 training curves before continuing
visualization.plot_combined_training_curves(
    teacher_history,
    phase1_label="Phase 1 – Frozen Base",
    phase2_label="(fine-tune not started yet)",
)

# Load best Phase-1 weights and unfreeze block5
frozen_model   = training_utils.load_model_compat(TEACHER_CHECKPOINT_PATH)
finetune_model = model_utils.unfreeze_vgg16_top_block(frozen_model, learning_rate=1e-5)

finetune_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=FINETUNE_CHECKPOINT_PATH,
    patience=5,
    monitor="val_auc",
)

finetune_history = finetune_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=finetune_callbacks,
    class_weight=class_weights,
    verbose=1,
)
print(f"Phase 2 complete. Best checkpoint saved to: {FINETUNE_CHECKPOINT_PATH}")


## Evaluation — Both VGG16 Models vs. Test Set

In [ ]:
# ── 6a+6b: Combined training curves + threshold tuning ──────────────────────

visualization.plot_combined_training_curves(
    teacher_history,
    finetune_history,
    phase1_label="Phase 1 – Frozen Base",
    phase2_label="Phase 2 – Fine-tune (block5)",
)

# Load best checkpoints for evaluation
frozen_best   = training_utils.load_model_compat(TEACHER_CHECKPOINT_PATH)
finetune_best = training_utils.load_model_compat(FINETUNE_CHECKPOINT_PATH)

# Tune decision threshold on validation set (maximises PNEUMONIA F1)
frozen_threshold,   frozen_val_f1,   _, _ = training_utils.tune_threshold(frozen_best,   val_ds)
finetune_threshold, finetune_val_f1, _, _ = training_utils.tune_threshold(finetune_best, val_ds)

print(f"VGG16 Frozen   — best val F1={frozen_val_f1:.3f}  @ threshold={frozen_threshold:.2f}")
print(f"VGG16 Finetuned — best val F1={finetune_val_f1:.3f}  @ threshold={finetune_threshold:.2f}")


In [ ]:
# ── 6c: Test set metrics for both VGG16 models ──────────────────────────────

frozen_metrics,   frozen_report,   y_true, frozen_prob,   frozen_pred   = training_utils.evaluate_model(frozen_best,   test_ds, frozen_threshold)
finetune_metrics, finetune_report, y_true, finetune_prob, finetune_pred = training_utils.evaluate_model(finetune_best, test_ds, finetune_threshold)

print("── VGG16 Frozen (Phase 1 best) ──────────────────────────────")
print(f"  Accuracy:    {frozen_metrics['accuracy']:.4f}")
print(f"  AUC:         {frozen_metrics['auc']:.4f}")
print(f"  Precision:   {frozen_metrics['precision']:.4f}")
print(f"  Recall:      {frozen_metrics['recall']:.4f}")
print(f"  F1:          {frozen_metrics['f1']:.4f}")
print(f"  Threshold:   {frozen_threshold:.2f}")

print("\n── VGG16 Fine-tuned (Phase 2 best) ─────────────────────────")
print(f"  Accuracy:    {finetune_metrics['accuracy']:.4f}")
print(f"  AUC:         {finetune_metrics['auc']:.4f}")
print(f"  Precision:   {finetune_metrics['precision']:.4f}")
print(f"  Recall:      {finetune_metrics['recall']:.4f}")
print(f"  F1:          {finetune_metrics['f1']:.4f}")
print(f"  Threshold:   {finetune_threshold:.2f}")

# Save both models with metadata
save_model_with_meta(frozen_best,   "vgg16_frozen",    frozen_metrics,   teacher_history,  {"lr": 1e-4, "freeze_base": True},                    frozen_threshold)
save_model_with_meta(finetune_best, "vgg16_finetuned", finetune_metrics, finetune_history, {"lr": 1e-5, "freeze_base": False, "unfrozen": "block5"}, finetune_threshold)


In [ ]:
# ── 6d: Side-by-side confusion matrices ─────────────────────────────────────

visualization.plot_confusion_matrices_grid([
    ("VGG16 Frozen",    y_true, frozen_pred),
    ("VGG16 Fine-tuned", y_true, finetune_pred),
])


In [ ]:
# ── 6e: Overlay ROC curves ───────────────────────────────────────────────────

visualization.plot_roc_curves([
    ("VGG16 Frozen",    y_true, frozen_prob,   frozen_threshold),
    ("VGG16 Fine-tuned", y_true, finetune_prob, finetune_threshold),
])


In [ ]:
# ── 6g: Qualitative predictions – best model ────────────────────────────────
# Show 9 test samples from the fine-tuned model with true label, prediction,
# and confidence score.

visualization.plot_sample_predictions(finetune_best, test_ds, finetune_threshold, n=9)


## Final Model Comparison

In [ ]:
# ── 6f: Final comparison table vs. paper targets ────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt

PAPER_TARGET = {
    "Model":     "Paper Target (VGG16 + class weights)",
    "Test Acc":  0.910,
    "Test AUC":  None,
    "Precision": 0.899,
    "Recall":    0.964,
    "F1":        0.931,
    "Threshold": "—",
    "Trainable Params": "—",
}

records = []
for model_name, label in [
    ("baseline_cnn",       "Baseline CNN"),
    ("baseline_cnn_tuned", "Baseline CNN (tuned HP)"),
    ("vgg16_frozen",       "VGG16 Frozen (Phase 1)"),
    ("vgg16_finetuned",    "VGG16 Fine-tuned (Phase 2)"),
]:
    meta = training_utils.load_model_meta(SAVE_DIR, model_name)
    if meta is None:
        continue
    m = meta["metrics"]
    records.append({
        "Model":      label,
        "Test Acc":   round(m.get("accuracy", 0), 4),
        "Test AUC":   round(m.get("auc", 0), 4),
        "Precision":  round(m.get("precision", 0), 4),
        "Recall":     round(m.get("recall", 0), 4),
        "F1":         round(m.get("f1", 0), 4),
        "Threshold":  round(meta.get("threshold", 0), 2),
        "Trainable Params": meta.get("model_params", {}).get("trainable", "—"),
    })

records.append(PAPER_TARGET)
comparison_df = pd.DataFrame(records).set_index("Model")

# ── Styled table ─────────────────────────────────────────────────────────────
metric_cols = ["Test Acc", "Test AUC", "Precision", "Recall", "F1"]
numeric_df  = comparison_df[metric_cols].apply(pd.to_numeric, errors="coerce")

def _highlight(col):
    best_val = col.dropna().max()
    colors = []
    for val in col:
        if pd.isna(val):
            colors.append("")
        elif val == best_val:
            colors.append("background-color: #c6efce; font-weight: bold")
        else:
            colors.append("")
    return colors

paper_row = comparison_df.index.get_loc(PAPER_TARGET["Model"])

def _mark_vs_paper(df):
    """Add check/cross suffix to metric cells based on paper target values."""
    paper_targets = {
        "Test Acc":  0.910,
        "Precision": 0.899,
        "Recall":    0.964,
        "F1":        0.931,
    }
    display = df.copy().astype(str)
    for col, target in paper_targets.items():
        if col not in display.columns:
            continue
        for idx in display.index:
            try:
                val = float(df.loc[idx, col])
                if "Paper" in str(idx):
                    continue
                symbol = " ✓" if val >= target else " ✗"
                display.loc[idx, col] = f"{val:.4f}{symbol}"
            except (ValueError, TypeError):
                pass
    return display

display_df = _mark_vs_paper(numeric_df)
print("=== Model Comparison vs. Paper Targets ===")
print(comparison_df[metric_cols + ["Threshold", "Trainable Params"]].to_string())

# ── Bar chart: AUC / Recall / Precision / F1 ─────────────────────────────────
plot_df = numeric_df.drop(index=PAPER_TARGET["Model"], errors="ignore")
paper_vals = numeric_df.loc[PAPER_TARGET["Model"]] if PAPER_TARGET["Model"] in numeric_df.index else None

ax = plot_df[["Test AUC", "Recall", "Precision", "F1"]].plot(
    kind="bar", figsize=(12, 5), colormap="tab10", edgecolor="white"
)
# Draw horizontal reference lines for paper targets
paper_lines = {"Recall": (0.964, "--"), "Precision": (0.899, ":"), "F1": (0.931, "-.")}
colors_map = {"Recall": "tab:orange", "Precision": "tab:green", "F1": "tab:red"}
for metric, (val, ls) in paper_lines.items():
    ax.axhline(val, linestyle=ls, color=colors_map[metric], linewidth=1.5,
               label=f"Paper {metric} = {val}")

ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Test Set Metrics  (dashed lines = paper targets)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()
